# Road PCI Intelligence — Step 3A: Model Pipeline, Sanity Checks & Smoke Test

This notebook implements the foundational semantic segmentation pipeline for **Road PCI Intelligence**:
1. **Architecture**: U-Net with ResNet-34 encoder pretrained on ImageNet (`segmentation-models-pytorch`), predicting 2 independent sigmoid logits: `[crack, pothole]`.
2. **Loss Formulation**: `MaskedMultiTaskLoss` combining BCEWithLogits and Soft Dice loss per channel, masking out gradients for un-supervised channels.
3. **Data Sampling**: Multi-source training with `SourceBalancedSampler` drawing 600 samples/epoch (200 crack500, 200 road_crack, 200 pothole).
4. **Sanity Check 1**: Unit test verification for loss gradients, hand-computed loss, zero-loss on unlabelled channels, and Dice extrema.
5. **Sanity Check 2**: Overfit test on 8 fixed samples (target Dice > 0.85).
6. **Sanity Check 3**: 3-epoch smoke test with augmentation, measuring train loss, per-source validation IoU and Dice, epoch time, and peak GPU memory.
7. **Sanity Check 4**: 6 validation prediction overlays per source saved to `/kaggle/working`.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, DEPENDENCIES & REPO REPLICATION
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# 1. Install required packages
print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

# 2. Clone GitHub repository for clean modular code and manifests
repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print("Pulling latest GitHub changes...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
import albumentations as A
import segmentation_models_pytorch as smp

from src.train import seed_everything, log_library_versions

seed_everything(42)
log_library_versions()

assert torch.cuda.is_available(), "CRITICAL: CUDA is not available! Enable GPU accelerator in Kaggle notebook settings."
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# 2. SANITY CHECK 1: UNIT TEST VERIFICATION FOR LOSS FUNCTIONS
# ==============================================================================
import unittest
import importlib.util

print("=" * 80)
print("RUNNING SANITY CHECK 1: UNIT TESTS FOR MASKED MULTI-TASK LOSS")
print("=" * 80)

test_path = repo_dir / 'tests/test_losses.py'
spec = importlib.util.spec_from_file_location("test_losses", str(test_path))
test_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_mod)

suite = unittest.TestLoader().loadTestsFromTestCase(test_mod.TestMaskedLosses)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

assert result.wasSuccessful(), "CRITICAL: Loss function unit tests failed!"
print("\n[PASS] Sanity Check 1 completed successfully.")


In [ ]:
# ==============================================================================
# 3. INITIALIZE DATA LOADERS & DATASET ROOTS
# ==============================================================================
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader

from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.datasets import RoadDistressDataset, SourceBalancedSampler

manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
manifest_df = pd.read_csv(manifest_path)
print(f"Loaded segmentation manifest: {len(manifest_df):,} records")

# Resolve dataset roots
dataset_roots = {}
for src in ['pothole', 'road_crack', 'crack500']:
    spec = DATASET_SPECS[src]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    assert root is not None and root.exists(), f"Could not find dataset root for {src}"
    dataset_roots[src] = root
    print(f"  • {src:10s} root: {root}")

# Build training dataset & balanced sampler
train_dataset = RoadDistressDataset(
    manifest_df=manifest_df,
    dataset_roots=dataset_roots,
    split='train',
    crop_size=512
)

train_sampler = SourceBalancedSampler(
    dataset=train_dataset,
    samples_per_epoch=600,
    seed=42
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True
)

# Build validation dataset & loader (full images, no augmentation, batch size 8)
val_dataset = RoadDistressDataset(
    manifest_df=manifest_df,
    dataset_roots=dataset_roots,
    split='val',
    crop_size=512
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\nTrain dataset: {len(train_dataset):,} images | Sampler: {len(train_sampler)} images/epoch ({len(train_loader)} batches/epoch)")
print(f"Val dataset:   {len(val_dataset):,} images ({len(val_loader)} batches)")


In [ ]:
# ==============================================================================
# 4. SANITY CHECK 2: OVERFIT TEST ON 8 SAMPLES (TARGET DICE > 0.85)
# ==============================================================================
import torch
from src.train import run_overfit_test

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
overfit_dice = run_overfit_test(train_dataset, device=device, num_steps=200)

assert overfit_dice > 0.85, f"Overfit test failed! Achieved {overfit_dice:.4f} <= 0.85"
print(f"\n[PASS] Sanity Check 2 passed: Final Mean Dice = {overfit_dice:.4f} > 0.85")


In [ ]:
# ==============================================================================
# 5. SANITY CHECK 3: 3-EPOCH SMOKE TEST WITH AUGMENTATION
# ==============================================================================
import time
import json
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler

from src.train import build_model, train_one_epoch, evaluate
from src.losses import MaskedMultiTaskLoss

print("=" * 80)
print("SANITY CHECK 3: 3-EPOCH SMOKE TEST WITH AUGMENTATION")
print("=" * 80)

model = build_model(device)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
total_steps = 3 * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
loss_fn = MaskedMultiTaskLoss(bce_weight=1.0, dice_weight=1.0)
scaler = GradScaler(enabled=True)

smoke_test_history = []

for epoch in range(1, 4):
    t0_epoch = time.time()
    torch.cuda.reset_peak_memory_stats(device)
    train_sampler.set_epoch(epoch)

    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        scaler=scaler,
        device=device,
        scheduler=scheduler
    )

    t_epoch = time.time() - t0_epoch
    peak_gpu_mem_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)

    # Validation evaluation
    val_results = evaluate(model=model, dataloader=val_loader, device=device)

    epoch_record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'epoch_time_s': t_epoch,
        'peak_gpu_mb': peak_gpu_mem_mb,
        'val_metrics': val_results
    }
    smoke_test_history.append(epoch_record)

    print(f"\n[EPOCH {epoch}/3] Completed in {t_epoch:.2f}s | Peak GPU: {peak_gpu_mem_mb:.1f} MB | Train Loss: {train_loss:.5f}")
    print(f"  • crack500:   Dataset IoU = {val_results.get('crack500_dataset_iou', 0.0):.4f} | Dataset Dice = {val_results.get('crack500_dataset_dice', 0.0):.4f} | Mean Img Dice = {val_results.get('crack500_mean_image_dice', 0.0):.4f}")
    print(f"  • road_crack: Dataset IoU = {val_results.get('road_crack_dataset_iou', 0.0):.4f} | Dataset Dice = {val_results.get('road_crack_dataset_dice', 0.0):.4f} | Mean Img Dice = {val_results.get('road_crack_mean_image_dice', 0.0):.4f}")
    print(f"  • pothole:    Dataset IoU = {val_results.get('pothole_dataset_iou', 0.0):.4f} | Dataset Dice = {val_results.get('pothole_dataset_dice', 0.0):.4f} | Mean Img Dice = {val_results.get('pothole_mean_image_dice', 0.0):.4f}")
    print(f"  • Overall Mean Source Dice: {val_results.get('mean_source_dice', 0.0):.4f}")

# Save metrics JSON to /kaggle/working
out_metrics_path = Path('/kaggle/working/smoke_test_metrics.json')
with open(out_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(smoke_test_history, f, indent=2)

print(f"\n[OK] Saved smoke test metrics to {out_metrics_path}")
print("[PASS] Sanity Check 3 completed successfully.")


In [ ]:
# ==============================================================================
# 6. SANITY CHECK 4: SAVE VALIDATION PREDICTION OVERLAYS
# ==============================================================================
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from src.train import save_prediction_overlays

output_dir = Path('/kaggle/working/overlays')
save_prediction_overlays(
    model=model,
    val_dataset=val_dataset,
    output_dir=output_dir,
    device=device,
    samples_per_source=6
)

# Display saved overlay figures in notebook
for src in ['crack500', 'road_crack', 'pothole']:
    fig_path = output_dir / f"val_overlays_{src}.png"
    if fig_path.exists():
        img = Image.open(fig_path)
        plt.figure(figsize=(12, 18))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Validation Overlays — {src}", fontsize=14, weight='bold', pad=10)
        plt.show()

print("\n[PASS] Sanity Check 4 completed successfully.")
